In [1]:
print(R.version$version.string)

[1] "R version 4.5.2 (2025-10-31 ucrt)"


In [3]:
suppressPackageStartupMessages({
    library("ibd")
    library("crossdes")
    library("lmerTest") # to print out p values with the outputs of lme4:: models
    library("lme4")
})
set.seed(2026 - 1 - 12)

## ___Power analysis___
-----------------

1. __Sympatric host ecotype – AM fungal community pairs will demonstrate superior adaptive fitness (measured by total plant biomass, plant height, PRLC & total length of hyphae in gram soil) compared to their allopatric counterparts__

2. __The extent of outsourcing reflected in the fine absorptive root traits (RD, SRL & RCT) will be the highest for each plant ecotype when paired with sympatric AM fungal community compared to pairings with foreign AM fungal communities.__

-----------------------

3. __Amongst the sympatric AM fungal-host ecotype pairs, the magnitude of this influence will be more pronounced where the hosts needed the AM fungi more (e.g. low phosphorus soils).__

4. __The fine absorptive root traits (collaboration gradient) will also be influenced by the AM fungal community composition in the soil samples, but this influence will be less pronounced compared to the provenance specific coadaptations.__

In [4]:
# have a look at the root trait data from Vin
themeda <- read.csv("../data/chapter3/vin_themeda_root_traits.csv", stringsAsFactors = TRUE)
head(themeda)

,ID,Accession,Rep,Weight..mg.,Weight..g.,Length.cm.,Length.m.,SurfArea.cm2.,AvgDiam.mm.,LenPerVol.cm.m3.,RootVolume.cm3.,SRL.m.g.,SRA.cm2.g.,RTD.gcm3.,SRL,SRA,RTD,Diameter
,<fct>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,Dalby-E,Dalby_QLD,E,368.4,0.3684,600.2167,6.002167,104.1133,0.5521,600.2167,1.437,16.29253,282.6094,0.2563674,16.29253,282.6094,0.2563674,0.5521
2,Dalby-F,Dalby_QLD,F,228.3,0.2283,812.4593,8.124593,126.7304,0.4965,812.4593,1.573,35.58735,555.1047,0.1451367,35.58735,555.1047,0.1451367,0.4965
3,Dalby-G,Dalby_QLD,G,279.9,0.2799,735.3292,7.353292,88.1581,0.3816,735.3292,0.841,26.27114,314.9628,0.3328181,26.27114,314.9628,0.3328181,0.3816
4,Dalby-H,Dalby_QLD,H,380.4,0.3804,662.1124,6.621124,134.1011,0.6447,662.1124,2.161,17.40569,352.5266,0.1760296,17.40569,352.5266,0.1760296,0.6447
5,55830-E,Mt Fox N Park_QLD,E,292.7,0.2927,1027.2178,10.272178,119.3707,0.3699,1027.2178,1.104,35.09456,407.8261,0.2651268,35.09456,407.8261,0.2651268,0.3699
6,55830-F,Mt Fox N Park_QLD,F,304.5,0.3045,721.0387,7.210387,113.3194,0.5003,721.0387,1.417,23.67943,372.1491,0.2148906,23.67943,372.1491,0.2148906,0.5003


In [6]:
sd(themeda$SRL)

[1] 10.6594

In [3]:
#-------------
# CONSTANTS
#-------------

PAIRS6X6 <- expand.grid(1:6, 1:6) # all possible combinations of soils and seeds
NROWS_PAIRS6X6 <- nrow(PAIRS6X6)
colnames(PAIRS6X6) <- c("seed", "soil") # update the column names

# parameters to specify a realistic random dist for SRL
MEAN_SRL <- 27.4254539108333 # mean(themeda$SRL)
STD_SRL <- 10.6594022650714 # sd(themeda$SRL)

NREPLICATES <- seq(from = 3, to = 12) # a realistic range of replicates we can afford to have in the experiment
EFFECT_SIZES <- seq(from = 0.1, to = 0.9, length.out = 30) # range of effect sizes to test
NITERATIONS <- 1000 # number of iterations to compute the proportion of times we had the defired effect?????

In [7]:
# a matrix to store our power values for each effect size and each number of reps
power_mat <- matrix(ncol = length(NREPLICATES), nrow = length(EFFECT_SIZES))

In [16]:
# trial model fitting with 

dummy <- setNames(expand.grid(seq(1, 6), seq(1, 6)), nm = c("soil", "seed"))
dummy["gsep"] <- mapply(dummy$soil == dummy$seed, FUN = function (b) ifelse(b, 'S', 'A'))
# do 8 reps for now
dummy <- dummy[rep(1:nrow(dummy), 8), ] # repeat each row 8 times
dummy["SRL"] <- rnorm(mean = MEAN_SRL, sd = STD_SRL, n = nrow(dummy))
dummy[dummy$soil == dummy$seed, ]$SRL <- dummy[dummy$soil == dummy$seed, ]$SRL * 1.35 # apply 35% effect to the sympatric pairs
mod <- lmerTest::lmer("SRL~gsep+(1|seed)+(1|soil)", data = dummy, REML = TRUE) # to get the p-values, use the lmer function from the lmerTest namespace instead of the nlme4 namespace

In [17]:
mod

Linear mixed model fit by REML ['lmerModLmerTest']
Formula: "SRL~gsep+(1|seed)+(1|soil)"
   Data: dummy
REML criterion at convergence: 2189.578
Random effects:
 Groups   Name        Std.Dev.
 seed     (Intercept)  0.3257 
 soil     (Intercept)  1.5324 
 Residual             10.8745 
Number of obs: 288, groups:  seed, 6; soil, 6
Fixed Effects:
(Intercept)        gsepS  
     28.678        7.037  

In [18]:
summary(mod)

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: "SRL~gsep+(1|seed)+(1|soil)"
   Data: dummy

REML criterion at convergence: 2189.6

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.8532 -0.6177  0.0278  0.6933  3.2492 

Random effects:
 Groups   Name        Variance Std.Dev.
 seed     (Intercept)   0.1061  0.3257 
 soil     (Intercept)   2.3482  1.5324 
 Residual             118.2558 10.8745 
Number of obs: 288, groups:  seed, 6; soil, 6

Fixed effects:
            Estimate Std. Error       df t value Pr(>|t|)    
(Intercept)  28.6780     0.9496   4.9074  30.199 9.16e-07 ***
gsepS         7.0370     1.7194 276.0012   4.093 5.60e-05 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Correlation of Fixed Effects:
      (Intr)
gsepS -0.302

In [19]:
anova(mod)

,Sum Sq,Mean Sq,NumDF,DenDF,F value,Pr(>F)
,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>
gsep,1980.8,1980.8,1,276.0012,16.75013,5.603237e-05


In [20]:
anova(mod)["gsep", "Pr(>F)"] # p-value of the fixed effect

[1] 5.603237e-05

In [ ]:
# three unusual things can happen with lmer(), 
# 1) insignificant random effects where we get a warning "boundary (singular) fit: see help('isSingular')"
# 2) the model may fail to converge with warning "Model failed to converge with max|grad| = 0.00410838 (tol = 0.002, component 1) See ?lme4::convergence and ?lme4::troubleshooting."
# 3) or with "Model failed to converge with 1 negative eigenvalue: -1.7e+00"

# all these warnings come from lme4's checkConv file - https://github.com/lme4/lme4/blob/02fc1b1a4f16c3960b6861860f7d917559988e02/R/checkConv.R
# even thogh its text reads "Warning" the first one is actually a MESSAGE, which need suppressMessages
# and the other two are warnings that can be suppressed with suppressWarnings

# 1) - https://stats.stackexchange.com/questions/512234/boundary-singular-fit-see-issingular
# 2) - https://stackoverflow.com/questions/75963799/mixed-model-with-lme4-failed-to-converge-with-maxgrad
# 3) - https://stats.stackexchange.com/questions/242109/model-failed-to-converge-warning-in-lmer

In [14]:
#-------------------------------------------------------------------------------------------------------------------------
# compute the power, given effect size and number of replicates while handling messages and warnings gracefully
#-------------------------------------------------------------------------------------------------------------------------

power <- function(dataset, niters, efsize, ssize, mn, stdev, bmask) {
    # dataset - the data to reference the variables in the formula against, will also be passed to lmerTest::lmer
    # niters - number of times to repeat the sampling & model fit
    # efsize - effect size
    # ssize - sample size
    # mn - mean for the normal dist to draw trait (SRL) values from
    # stdev - standard deviation for the normal dist to draw trait (SRL) values from
    # bmask - boolean mask specifying the sympatric records

    pvalues <- vector(mode = "numeric", length = niters)
    
    for(i in 1:niters) {
        
        repeat {
            
            failed_to_converge <- FALSE # cannot have this in the outer scope (for loop's scope) because this needs to be reset for each iteration to avoid infinite looping
            
            withCallingHandlers( # look up for reference - https://adv-r.hadley.nz/conditions.html
                warning = function(condition) { # run this function when a warning is signalled
                    # probable causes for this handler being called - 
                    # 1) Model failed to converge with 1 negative eigenvalue .............
                    # 2) Model failed to converge with max|grad| .........
                    failed_to_converge <<- TRUE
                    # more on invokeRestart at https://docs.tibco.com/pub/enterprise-runtime-for-R/6.1.0/doc/html/Language_Reference/base/conditions.html
                    invokeRestart(r="muffleWarning") # has a simple recovery strategy: “Suppress the warning”. it consumes the warning (so it does not “bubble up” to higher function call levels) and resumes the execution.
                },
                message = function(condition) { # run this function when a message is signalled
                     # probable cause for this handler getting called - "boundary (singular) fit: see help('isSingular')"
                    invokeRestart(r="muffleMessage") # suppress the messages - USING "muffleWarning" WON'T (DIDN'T) HELP HERE BECAUSE THIS HANDLER IS FOR MESSAGES
                },
                # the block of code to be executed within the control of the handlers (i.e expr)
                {
                    dataset["SRL"] <- rnorm(mean = mn, sd = stdev, n = ssize) # randomly populate the SRL, using the specified params
                    dataset$SRL[bmask] <- dataset$SRL[bmask] * (1 + efsize) # apply the effect for sympatric records
                    mixmod <- lmerTest::lmer(formula = "SRL~gsep+(1|seed)+(1|soil)", data = dataset) # fit the model to the data, if we do use lme4::lmer(), the next line will raise an error because the output of lme4::lmer does not include p values
                    # if a warning is emitted from the above line, the handler will set failed_to_converge to TRUE
                    pvalues[i] <- anova(mixmod)["gsep", "Pr(>F)"] # this line will always be executed, this is where the control returns after a warning or message is caught
                }
            )
            
            if (!failed_to_converge) break # if the model converged succesfully (no warnings from lme4::lmer), break out the loop, else continue to resample and model fit.
            
        }
    }
    mean(pvalues < 0.05) # return fraction of p-values that are less than 0.05 (which is our power????)
}

In [15]:
#---------------------
# simulation loop
#---------------------

tm <- Sys.time()

for (i in seq_along(NREPLICATES)) {
        data <- PAIRS6X6[rep(1:NROWS_PAIRS6X6, NREPLICATES[i]), ] # expand the PAIRS6X6 dataframe such that each row gets repcilated NREPLICATES[i] times
        bmask_sympatric <- (data$soil == data$seed) # a boolean mask for sympatric records in the dataset
        # new column based on the boolean mask => geographical separation, convenient for specifying the model formula
        # without the result of lapply() wrapped inside an unlist(), the new column is annealed as a dataframe instead of a vector, which raises an exception when fitting the model or just use mapply()
        data["gsep"] <- mapply(bmask_sympatric, FUN = function (b) ifelse(b, 'S', 'A'))
        ss <- nrow(data) # sample size for the rnorm function - DO NOT CONFUSE THIS WITH THE NUMBER OF REPLICATES
    
    for (j in seq_along(EFFECT_SIZES)) {            
        # allopatric vs sympatric becomes our fixed effect => gsep
        # seed and soil origins become our (non nested) random effects
        # https://stats.stackexchange.com/questions/674034/statistical-test-for-significance-of-mean-differences-between-two-groups
        # in the matrix called power_mat, rows are effect sizes and columns are number of reps
        power_mat[j, i] <- power(dataset = data, niters = NITERATIONS, efsize = EFFECT_SIZES[j], ssize = ss, mn = MEAN_SRL, stdev = STD_SRL, bmask = bmask_sympatric) # power - averaged over 1000 random samplings
    }
}

tm <- Sys.time() - tm

In [16]:
tm # Time difference of 2.279102 hours - not that bad huh???

Time difference of 2.279102 hours

In [35]:
# deparse.level = 0 is needed to avoid cbind() using the vector's name as a column name
rbind(c(NA, NREPLICATES), cbind(EFFECT_SIZES, power_mat, deparse.level = 0)) # first row - number of replicates, first column - effect sizes - then just serialize it

NA,3.000,4.000,5.000,6.000,7.000,8.000,9.000,10.000,11.000,12.000
0.1000000,0.185,0.232,0.257,0.290,0.326,0.356,0.394,0.468,0.477,0.497
0.1275862,0.261,0.321,0.365,0.435,0.495,0.538,0.570,0.621,0.655,0.666
0.1551724,0.372,0.397,0.482,0.560,0.624,0.656,0.706,0.770,0.799,0.837
0.1827586,0.435,0.553,0.606,0.678,0.772,0.786,0.832,0.891,0.884,0.911
0.2103448,0.541,0.626,0.709,0.786,0.844,0.883,0.889,0.944,0.950,0.963
0.2379310,0.625,0.697,0.818,0.872,0.900,0.948,0.957,0.978,0.977,0.988
0.2655172,0.693,0.787,0.874,0.922,0.962,0.969,0.981,0.989,0.991,0.994
0.2931034,0.734,0.836,0.903,0.955,0.969,0.989,0.995,0.997,0.999,0.999
0.3206897,0.826,0.890,0.948,0.969,0.984,0.998,0.993,0.998,1.000,1.000
0.3482759,0.863,0.925,0.966,0.978,0.996,0.997,0.998,0.999,1.000,1.000


In [39]:
write.table(x = rbind(c(NA, NREPLICATES), cbind(EFFECT_SIZES, power_mat, deparse.level = 0)), file = "./power.csv", row.names = FALSE, col.names = FALSE, sep = ',') # col.names = FALSE is not allowed by write.csv()

## ___Balanced Incomplete Block Design (BIBD)___
--------------------------------

In [3]:
# reference - https://people.math.ethz.ch/~meier/teaching/anova/incomplete-block-designs.html

In [7]:
# let's say that we need 36 reps for each allopatric and sympatric groups
# the sympatric 36 can be easily divided between the 6 pairs (6 reps per each pair)
# how do we divide the other 36 between the 30 potential pairs????

In [ ]:
# treat the remaining 30 pairs as 30 individual units, numbered 1 to 30
# 

In [ ]:
# treat the seeds as blocks => 6 blocks
# each block will be tested against 6 soil treatments (in a complete block design)
# say that in an IBD, each seed only gets three soil treatments

In [21]:
# - v: number of treatments
# - b: number of blocks
# - r: number of replicates (across all blocks)
# - k: number of experimental units per block
# - lambda: lambda

ibd::bibd(v=5, b=5, k=3, r=3, lambda=2)

[1] "parameters do not satisfy necessary conditions"

In [25]:
ibd::ibd(v=6, b=6, k=4)

$v
[1] 6

$b
[1] 6

$k
[1] 4

$NNP
     [,1] [,2] [,3] [,4] [,5] [,6]
[1,]    4    2    2    2    3    3
[2,]    2    4    3    2    2    3
[3,]    2    3    4    3    2    2
[4,]    2    2    3    4    3    2
[5,]    3    2    2    3    4    2
[6,]    3    3    2    2    2    4

$N
     [,1] [,2] [,3] [,4] [,5] [,6]
[1,]    1    1    1    0    0    1
[2,]    0    0    1    1    1    1
[3,]    0    1    0    1    1    1
[4,]    1    1    0    1    1    0
[5,]    1    1    1    0    1    0
[6,]    1    0    1    1    0    1

$design
        [,1] [,2] [,3] [,4]
Block-1    1    4    5    6
Block-2    1    3    4    5
Block-3    1    2    5    6
Block-4    2    3    4    6
Block-5    2    3    4    5
Block-6    1    2    3    6

$conc.mat
     [,1] [,2] [,3] [,4] [,5] [,6]
[1,]    4    2    2    2    3    3
[2,]    2    4    3    2    2    3
[3,]    2    3    4    3    2    2
[4,]    2    2    3    4    3    2
[5,]    3    2    2    3    4    2
[6,]    3    3    2    2    2    4

$A.Effici